## RAG Application using typesense

In [1]:
import typesense
import os
from dotenv import load_dotenv


In [2]:

load_dotenv("../.env")
typesense_api_key = os.getenv("TYPESENSE_ADMIN_API_KEY")

In [ ]:
client=typesense.Client({
    'nodes':[{
        'host':'13jn9hwbpgqzemtsp-1.a2.typesense.net',
        'port':'443',
        'protocol':'https'
    }],
    'api_key':typesense_api_key,
    'connection_timeout_seconds': 2
})

books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

In [4]:
with open('../data/books.jsonl','r',encoding='utf-8') as jsonl_file:
    data=jsonl_file.read()
    client.collections['books'].documents.import_(data)


In [8]:
search_parameters={
    'q':"harry porter",
    'query_by':"title,authors",
    'sort_by': 'ratings_count:desc'
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451402721626233,
   'text_match_info': {'best_field_score': '2211864313857',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451402721626233',
    'tokens_matched': 2,
    'typo_prefix_score': 2}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

### Langchain + Typsense + Groq LLM + RAG Application

In [3]:
### Langchain + Typsense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22956\1884500311.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\Ai\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
groq_api_key=os.getenv("GROQ_API_KEY")

In [5]:
loader= TextLoader("../data/text_files/test.txt")
print(loader)

documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs= text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22956\4292438481.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22956\4292438481.py:8: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6901.32it/s]


In [6]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": "13jn9hwbpgqzemtsp-1.a2.typesense.net",
        "port": "443",
        "protocol": "https",
        "typesense_api_key": typesense_api_key,
        "typesense_collection_name": "lang_chain",
    },
)

In [7]:
query ="What is artificial intelligence?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

# Artificial Intelligence and Machine Learning

Artificial Intelligence (AI) and Machine Learning (ML) have become two of the most influential technologies of the modern era. They are transforming industries, improving decision-making, and enabling computers to perform tasks that once required human intelligence. From healthcare and finance to education and transportation, AI and ML are driving innovation and creating new opportunities for businesses and individuals alike.

Artificial Intelligence is a broad field of computer science focused on building systems capable of performing tasks such as reasoning, learning, problem-solving, language understanding, and decision-making. Machine Learning is a subset of AI that enables computers to learn patterns from data instead of relying solely on explicitly programmed rules. By analyzing large volumes of historical data, ML models can make predictions, classify information, and continuously improve their performance over time.


In [10]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x00000191B2EEDBE0>, search_kwargs={})

In [9]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': '../data/text_files/test.txt'}, page_content='# Artificial Intelligence and Machine Learning\n\nArtificial Intelligence (AI) and Machine Learning (ML) have become two of the most influential technologies of the modern era. They are transforming industries, improving decision-making, and enabling computers to perform tasks that once required human intelligence. From healthcare and finance to education and transportation, AI and ML are driving innovation and creating new opportunities for businesses and individuals alike.\n\nArtificial Intelligence is a broad field of computer science focused on building systems capable of performing tasks such as reasoning, learning, problem-solving, language understanding, and decision-making. Machine Learning is a subset of AI that enables computers to learn patterns from data instead of relying solely on explicitly programmed rules. By analyzing large volumes of historical data, ML models can make predictions, classify in